# Sample poses from a trained model





Pick any trained model of the model zoo (RSGM runs, flows converted from `SO3PriorTrainer`, ...), draw poses from it and look at them, next to real poses of the test split.

**Rendering.** Meshes are drawn with PyVista (`hp.visualization.pv_render`, offscreen; the pyrender backend of body_visualizer does not run on macOS). That needs the SMPL-X body model `smplx/SMPLX_NEUTRAL.npz` in the `body_models` folder of `config/paths.yaml` (download it from https://smpl-x.is.tue.mpg.de/downloads; it is licensed, so it is not in the repository). Without it the poses are drawn as skeletons (forward kinematics of SMPL, which only needs `smpl_skeleton.npz`), from the front and from the side.

The models speak 19 joint rotations (`AmassJoints4Learning`); for the body model the two toe joints, which are not in the data (always the rest pose in our AMASS), are set to the rest pose.

Sections: 1. the models there are · 2. one model: sample, log-likelihood, pictures · 3. real poses for comparison · 4. an interactive panel.

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import roma
import torch
from omegaconf import OmegaConf

from hp.data.amass_cfg import AmassJoints4Learning
from hp.evaluation.data import load_reference
from hp.models.io import MANIFEST, load_model
from hp.utils.paths import get_path
from hp.utils.skeleton_forward_kinematics import SkelForwardKinematics, perform_forward_kinematics

# ---- parameters ----------------------------------------------------------------------
EXPERIMENTS = "~/data/experiments"      # searched for trained models (folders with model.json)
INCLUDE_TRIALS = False                  # also list the trials of the hyperparameter search and the checkpoints
MODEL = "rsgm_tuned_t23/run_0001"       # a folder below EXPERIMENTS (see the table of section 1), or an absolute path
WEIGHTS = "weights_ema"                 # weights_ema (if the run has them) or weights
N_SAMPLES = 8
SEED = 0
SAMPLE_STEPS = None                     # steps of the sampler (RSGM); None: the model's own setting
LIKELIHOOD_STEPS = 200                  # ODE steps of RSGM's log-likelihood; 0: do not compute it
DEVICES = {"rsgm": "mps", "so3_flow": "cpu"}   # by model type; others: cpu
VIEWS = (0, 90)                         # azimuths (degrees) of the mesh renders; skeletons: front and side
TEST_ROTATIONS = "~/data/AMASS_rotations"     # real poses for comparison (the official test split)
# ----------------------------------------------------------------------------------------

expand = lambda p: Path(p).expanduser()
EXPERIMENTS = expand(EXPERIMENTS)

## 1. The models there are

Every folder below `EXPERIMENTS` with a `model.json` (a checkpoint of the model zoo), with its type, size and the data it was trained on (from its `config.yaml`, if it is a run of `scripts/train_model.py`).

In [ ]:
def model_table(root, include_trials=False):
    rows = []
    for manifest in sorted(root.rglob(MANIFEST)):
        folder = manifest.parent
        rel = folder.relative_to(root)
        if not include_trials and ("checkpoint_" in str(rel) or str(rel).startswith("rsgm_hpo")):
            continue
        args = json.loads(manifest.read_text())["build_args"]
        config = folder / "config.yaml"
        data, steps = "-", "-"
        if config.is_file():
            c = OmegaConf.load(config)
            data = str(c.get("data", {}).get("rotations", "-"))
            steps = c.get("train", {}).get("max_iterations", "-")
        hidden = args.get("hidden")
        rows.append(dict(model=str(rel), type=args["name"], net=f"{hidden[0]} x {len(hidden)}" if hidden else "-",
                         trained_on=data.replace("~/data/", ""), steps=steps,
                         ema=(folder / "weights_ema.pt").is_file()))
    return pd.DataFrame(rows)


models = model_table(EXPERIMENTS, INCLUDE_TRIALS)
pd.set_option("display.width", 200)
print(models.to_string(index=False))

## 2. Sample from one model

Rendering helpers first: the 19 rotations of a pose become the 63 axis-angle numbers of the 21 body joints of SMPL-X (`pose_body`), which the body model turns into a mesh. Without the body model, the joints of SMPL's skeleton are drawn instead.

In [ ]:
PARENTS = SkelForwardKinematics().parents[:22]
BODY_MODEL = Path(get_path("body_models")) / "smplx" / "SMPLX_NEUTRAL.npz"


def to_pose_body(poses):
    # [N, 19, 3, 3] rotation matrices -> [N, 63]: axis-angle of the 21 body joints, the toes (10, 11) at rest
    n = len(poses)
    pose_body = torch.zeros(n, 21, 3)
    pose_body[:, [j.value - 1 for j in AmassJoints4Learning]] = roma.rotmat_to_rotvec(poses.float().cpu())
    return pose_body.reshape(n, 63)


renderer = None
if BODY_MODEL.is_file():
    from hp.visualization.render_tools import HumanRender
    renderer = HumanRender()
    print(f"meshes with PyVista, body model {BODY_MODEL}")
else:
    print(f"no body model at {BODY_MODEL}: skeletons instead of meshes")


def draw_skeleton(ax, joints, view, color="C0"):
    a, b = (0, 1) if view == "front" else (2, 1)
    for j in range(1, 22):
        p = PARENTS[j]
        ax.plot([joints[j, a], joints[p, a]], [joints[j, b], joints[p, b]], color=color, lw=2.2, solid_capstyle="round")
    ax.scatter(joints[:22, a], joints[:22, b], s=6, color=color)
    ax.set_aspect("equal")
    ax.set_xlim(-1.0, 1.0)
    ax.set_ylim(-1.25, 0.85)
    ax.axis("off")


def crop_margins(images, margin=0.03):
    # Crop the background around the figures of all rendered images [N, views, h, w, 3] with the same box,
    # so that the poses keep their relative size. The background is the colour of the corner pixel.
    background = images[:, :, :1, :1].astype(int)
    used = (np.abs(images.astype(int) - background) > 8).any(axis=-1).any(axis=(0, 1))   # [h, w]
    rows, cols = np.where(used.any(axis=1))[0], np.where(used.any(axis=0))[0]
    if len(rows) == 0:
        return images
    h, w = used.shape
    pad_r, pad_c = int(margin * h), int(margin * w)
    r0, r1 = max(rows[0] - pad_r, 0), min(rows[-1] + pad_r + 1, h)
    c0, c1 = max(cols[0] - pad_c, 0), min(cols[-1] + pad_c + 1, w)
    return images[:, :, r0:r1, c0:c1]


VIEW_SETS = {"both": (0, 1), "front": (0,), "side": (1,)}   # indices into VIEWS (the azimuths) / (front, side) of the skeletons


def show_poses(poses, titles=None, color="C0", suptitle=None, ncols=None, view="both", marked=(), caption=None):
    # Draw poses [N, 19, 3, 3], `ncols` poses per block (None: all in one block); the blocks are stacked.
    # Meshes if the body model is there, else skeletons. `view`: both, front or side. Every pose has a frame
    # around its views, red for the poses with an index in `marked`. `caption`: a line
    # in the lower left corner (model, seed, ...). Returns the figure.
    n = len(poses)
    titles = titles or [""] * n
    ncols = min(ncols or n, n)
    blocks = -(-n // ncols)
    chosen = VIEW_SETS[view]
    views = len(chosen)
    if renderer is not None:
        images = renderer.render_images(to_pose_body(poses), azimuths_deg=[VIEWS[k] for k in chosen])  # [N, views, h, w, 3]
        images = crop_margins(images)
        cell_w = 1.9
    else:
        joints = perform_forward_kinematics(poses.float().cpu(), AmassJoints4Learning).numpy()
        cell_w = 1.6
    # rows: the views of a block, then a spacer row (room for the next titles) between the blocks
    rows = blocks * (views + 1) - 1
    ratios = ([1.0] * views + [0.3]) * blocks
    ratios = ratios[:rows]
    fig, axes = plt.subplots(rows, ncols, figsize=(cell_w * ncols, 2.05 * views * blocks + 0.65 * (blocks - 1) + 0.4),
                             gridspec_kw={"height_ratios": ratios}, squeeze=False)
    row = lambda b, r: b * (views + 1) + r
    for ax in axes.flat:
        ax.axis("off")
    for i in range(n):
        b, c = divmod(i, ncols)
        for r, k in enumerate(chosen):
            ax = axes[row(b, r), c]
            if renderer is not None:
                ax.imshow(images[i, r])
            else:
                draw_skeleton(ax, joints[i], ("front", "side")[k], color)
            if views == 2:                           # the two views of a pose close together
                ax.set_anchor("S" if r == 0 else "N")
        axes[row(b, 0), c].set_title(titles[i], fontsize=8)
    if renderer is None:
        for b in range(blocks):
            for r, k in enumerate(chosen):
                axes[row(b, r), 0].text(-1.0, 0.95, ("front", "side")[k], fontsize=8, color="0.4")
    if suptitle:
        fig.suptitle(suptitle, fontsize=10)
    fw0, fh0 = fig.get_size_inches()
    fig.tight_layout(h_pad=0.3, w_pad=1.6, rect=(0, 0.3 / fh0 if caption else 0, 1, 1))   # room for the caption below
    if caption:
        fig.text(0.005, 0.1 / fh0, caption, fontsize=7, color="0.35", ha="left", va="bottom")
    fig.canvas.draw()                                # applies the aspect: get_position is then the box of the picture
    fw, fh = fig.get_size_inches()                   # a frame around each pose, with its views and its title
    px, py_top, py_bot = 0.06 / fw, 0.34 / fh, 0.06 / fh
    for i in range(n):
        b, c = divmod(i, ncols)
        top, bottom = axes[row(b, 0), c].get_position(), axes[row(b, views - 1), c].get_position()
        x0, x1, y0, y1 = top.x0 - px, top.x1 + px, bottom.y0 - py_bot, top.y1 + py_top
        red = i in marked
        fig.add_artist(Rectangle((x0, y0), x1 - x0, y1 - y0, transform=fig.transFigure, fill=False,
                                 edgecolor="red" if red else "0.7", linewidth=2.0 if red else 0.8))
    plt.show()
    return fig


def load(name, weights=WEIGHTS):
    folder = Path(name) if Path(name).is_absolute() else EXPERIMENTS / name
    kind = json.loads((folder / MANIFEST).read_text())["build_args"]["name"]
    if not (folder / f"{weights}.pt").is_file():
        weights = "weights"
    return load_model(folder, device=DEVICES.get(kind, "cpu"), weights=weights), weights


def sample(model, n, seed, sample_steps=None, likelihood_steps=LIKELIHOOD_STEPS):
    # n poses from the model, and their log-likelihoods (None if not computed). Returns poses on the cpu.
    if sample_steps is not None and hasattr(model, "sample_steps"):
        model.sample_steps = sample_steps
    torch.manual_seed(seed)
    start = time.perf_counter()
    with torch.no_grad():
        poses = model.sample(n).float().cpu()
    seconds = time.perf_counter() - start
    ll = None
    if likelihood_steps and model.has_log_prob:
        if hasattr(model, "likelihood_steps"):
            model.likelihood_steps = likelihood_steps
        with torch.no_grad():
            ll = model.log_prob(poses.to(model.device)).float().cpu().numpy()
    return poses, ll, seconds

In [ ]:
model, used_weights = load(MODEL)
poses, ll, seconds = sample(model, N_SAMPLES, SEED, SAMPLE_STEPS)
print(f"{MODEL} ({model.name}, {model.num_parameters():,} parameters, {used_weights}): {N_SAMPLES} poses in {seconds:.2f} s")
titles = [f"#{i}" + (f"\nlog p {ll[i]:.0f}" if ll is not None else "") for i in range(N_SAMPLES)]
show_poses(poses, titles, color="C1", suptitle=f"samples of {MODEL}, seed {SEED}")
if ll is not None:
    print("log-likelihood of the samples:", np.round(ll, 1), f"(mean {ll.mean():.1f})")

## 3. Real poses for comparison

Random poses of the official test split (Transitions + SSM, all frames), with their log-likelihood under the chosen model.

In [ ]:
test = load_reference("test", TEST_ROTATIONS, "original")
real = test[torch.randperm(len(test), generator=torch.Generator().manual_seed(SEED))[:N_SAMPLES]]
real_ll = None
if LIKELIHOOD_STEPS and model.has_log_prob:
    with torch.no_grad():
        real_ll = model.log_prob(real.to(model.device)).float().cpu().numpy()
titles = [f"test" + (f"\nlog p {real_ll[i]:.0f}" if real_ll is not None else "") for i in range(N_SAMPLES)]
show_poses(real, titles, color="C0", suptitle=f"real test poses, log p under {MODEL}")

## 4. Interactive panel

Choose a model, the number of poses, the seed and the sampler's steps, and press **Sample**. Models are loaded once and kept.

**Views** (front, side or both) and **per row** only change the picture: the same samples are drawn again as soon as you change them (or press **Redraw**), nothing is sampled anew.

**Wrong poses.** Type the number of a pose (or several, \"3, 5 7\") or pick one from the list and press **Add** (**Remove** and **Clear** take them out). Marked poses get a red frame the next time the picture is drawn (**Redraw**). **Save wrong poses** writes `<name>_wrong.json` to the folder: the date, model, weights, seed, number of poses and sampler steps (so the same samples can be drawn again), the indices, and the rotation matrices of the wrong poses. **Show saved** draws the poses of a saved list. New samples clear the marks.

Every picture has a caption in its lower left corner: model, weights, seed, number of poses and sampler steps, so that it can be reproduced.

**Save picture** writes the whole figure as a PNG to the folder (default `~/data/sample_poses`). The name is filled in after every sample as `<model>_<number>` (the first number that is free in the folder); edit it if you like. Nothing is ever overwritten: a `_2`, `_3`, ... is added.

In [ ]:
import datetime

import ipywidgets as widgets
from IPython.display import display

SAVE_DIR = "~/data/sample_poses"        # default folder of the saved pictures and of the lists of wrong poses

cache = {}
last = {"fig": None}                    # the last samples (poses, titles, header, parameters) and their figure
wrong = set()                           # indices of the poses marked as wrong

L = lambda w: widgets.Layout(width=w)
model_box = widgets.Dropdown(options=list(models.model), value=MODEL if MODEL in list(models.model) else None,
                             description="model", layout=L("520px"))
n_box = widgets.BoundedIntText(value=N_SAMPLES, min=1, max=256, description="poses", layout=L("160px"))
cols_box = widgets.BoundedIntText(value=min(N_SAMPLES, 8), min=1, max=16, description="per row", layout=L("160px"))
seed_box = widgets.IntText(value=SEED, description="seed", layout=L("160px"))
steps_box = widgets.IntText(value=100, description="steps", layout=L("160px"))
ll_box = widgets.Checkbox(value=True, description="log-likelihood")
view_box = widgets.ToggleButtons(options=["both", "front", "side"], description="views")
button = widgets.Button(description="Sample", button_style="primary")
redraw_button = widgets.Button(description="Redraw", icon="refresh", tooltip="draw the same samples again, with the current views, poses per row and marks")
dir_box = widgets.Text(value=SAVE_DIR, description="folder", layout=L("520px"))
name_box = widgets.Text(value="", description="name", placeholder="default: model name and a number", layout=L("520px"))
save_button = widgets.Button(description="Save picture", icon="save")
# wrong poses: type indices ("3", "3, 5 7") or pick one from the list, Add / Remove; the red frames show up on Redraw
pick_box = widgets.Combobox(options=[], placeholder="pose number(s)", description="wrong", ensure_option=False, layout=L("260px"))
add_button = widgets.Button(description="Add", icon="plus")
remove_button = widgets.Button(description="Remove", icon="minus")
clear_button = widgets.Button(description="Clear")
wrong_label = widgets.HTML()
save_wrong_button = widgets.Button(description="Save wrong poses", icon="save")
saved_box = widgets.Dropdown(options=[], description="saved", layout=L("420px"))
show_saved_button = widgets.Button(description="Show saved", icon="eye")
out = widgets.Output()
msg = widgets.Output()


def default_name(model_name, folder, suffix=".png"):
    # <model>_<number>: the model's folder with "/" as "_", the first number that is still free in the folder
    slug = model_name.replace("/", "_")
    k = 1
    while (folder / f"{slug}_{k:03d}{suffix}").exists():
        k += 1
    return f"{slug}_{k:03d}"


def free_path(folder, name, suffix):
    # folder / name+suffix, with _2, _3, ... if it exists: nothing is overwritten
    folder.mkdir(parents=True, exist_ok=True)
    path, k = folder / f"{name}{suffix}", 2
    while path.exists():
        path = folder / f"{name}_{k}{suffix}"
        k += 1
    return path


def say(text):
    with msg:
        msg.clear_output(wait=True)
        print(text)


def draw():
    # draw the last samples again: new views / poses per row / marks, the same poses
    with out:
        out.clear_output(wait=True)
        if "poses" not in last:
            print("sample first")
            return
        print(last["header"])
        q = last["params"]
        caption = (f"{q['model']} [{q['weights']}]   seed {q['seed']}   {q['n']} poses   {q['sample_steps']} sampler steps"
                   + (f"   (saved {q['date']})" if "date" in q else ""))
        last["fig"] = show_poses(last["poses"], last["titles"], color="C1", ncols=cols_box.value, view=view_box.value,
                                 marked={i for i in wrong if i < len(last["poses"])}, caption=caption)


def refresh_wrong():
    wrong_label.value = ("<b>wrong:</b> " + (", ".join(f"#{i}" for i in sorted(wrong)) or "none")
                         + " <span style='color:gray'>(red frames after Redraw)</span>")


def refresh_saved():
    folder = expand(dir_box.value)
    files = sorted(f.name for f in folder.glob("*_wrong*.json")) if folder.is_dir() else []
    saved_box.options = files


def on_sample(_):
    with out:
        out.clear_output(wait=True)
        if model_box.value is None:
            print("choose a model")
            return
        if model_box.value not in cache:
            print(f"loading {model_box.value} ...")
            cache[model_box.value] = load(model_box.value)
        m, used_weights = cache[model_box.value]
        p, l, sec = sample(m, n_box.value, seed_box.value, steps_box.value, LIKELIHOOD_STEPS if ll_box.value else 0)
        last.update(poses=p, ll=l, model=model_box.value,
                    titles=[f"#{i}" + (f"\nlog p {l[i]:.0f}" if l is not None else "") for i in range(len(p))],
                    header=f"{model_box.value}: requested {n_box.value}, drew {len(p)} poses in {sec:.2f} s"
                           + (f", mean log p {l.mean():.1f}" if l is not None else ""),
                    params=dict(model=model_box.value, weights=used_weights, seed=seed_box.value, n=n_box.value,
                                sample_steps=steps_box.value, likelihood_steps=LIKELIHOOD_STEPS if ll_box.value else 0))
        wrong.clear()
        refresh_wrong()
        pick_box.options = [str(i) for i in range(len(p))]
        pick_box.value = ""
        name_box.value = default_name(model_box.value, expand(dir_box.value))
    draw()


def parse_indices(text):
    return [int(t) for t in text.replace(",", " ").replace("#", " ").split()]


def on_add(_):
    if "poses" not in last:
        return say("sample first")
    try:
        found = parse_indices(pick_box.value)
    except ValueError:
        return say(f"cannot read pose numbers from {pick_box.value!r}")
    bad = [i for i in found if not 0 <= i < len(last["poses"])]
    wrong.update(i for i in found if i not in bad)
    refresh_wrong()
    pick_box.value = ""
    say(f"no such poses: {bad}" if bad else "")


def on_remove(_):
    try:
        wrong.difference_update(parse_indices(pick_box.value))
    except ValueError:
        return say(f"cannot read pose numbers from {pick_box.value!r}")
    refresh_wrong()
    pick_box.value = ""
    say("")


def on_clear(_):
    wrong.clear()
    refresh_wrong()
    say("")


def on_save(_):
    if last["fig"] is None:
        return say("sample first")
    folder = expand(dir_box.value)
    name = (name_box.value.strip() or default_name(last["model"], folder)).removesuffix(".png")
    path = free_path(folder, name, ".png")
    last["fig"].savefig(path, dpi=200, bbox_inches="tight")
    say(f"saved {path}")


def on_save_wrong(_):
    # the wrong poses with what is needed to sample the same ones again (model, weights, seed, n, steps) and
    # the rotation matrices themselves, in case the sampling does not repeat exactly
    if "poses" not in last or not wrong:
        return say("nothing to save: sample, then add wrong poses")
    idx = sorted(wrong)
    folder = expand(dir_box.value)
    name = (name_box.value.strip() or default_name(last["model"], folder, "_wrong.json")).removesuffix(".png")
    path = free_path(folder, name + "_wrong", ".json")
    data = dict(date=datetime.datetime.now().isoformat(timespec="seconds"), **last["params"], wrong=idx,
                log_p=None if last["ll"] is None else [float(last["ll"][i]) for i in idx],
                rotations=last["poses"][idx].tolist())     # [k, 19, 3, 3], the joints of AmassJoints4Learning
    path.write_text(json.dumps(data, indent=1))
    refresh_saved()
    say(f"saved {len(idx)} wrong poses to {path}")


def on_show_saved(_):
    if not saved_box.value:
        return say("no saved list of wrong poses in the folder")
    data = json.loads((expand(dir_box.value) / saved_box.value).read_text())
    ll = data.get("log_p")
    last.update(poses=torch.tensor(data["rotations"]), ll=None, model=data["model"], params={k: v for k, v in data.items() if k not in ("rotations", "log_p", "wrong")},
                titles=[f"#{i}" + (f"\nlog p {ll[j]:.0f}" if ll else "") for j, i in enumerate(data["wrong"])],
                header=f"{data['model']}, seed {data['seed']}, n {data['n']}: {len(data['wrong'])} wrong poses of {data['date']}")
    wrong.clear()
    refresh_wrong()
    pick_box.options = []
    draw()


button.on_click(on_sample)
redraw_button.on_click(lambda _: draw())
save_button.on_click(on_save)
add_button.on_click(on_add)
remove_button.on_click(on_remove)
clear_button.on_click(on_clear)
save_wrong_button.on_click(on_save_wrong)
show_saved_button.on_click(on_show_saved)
cols_box.observe(lambda _: draw() if "poses" in last else None, names="value")   # views and poses per row
view_box.observe(lambda _: draw() if "poses" in last else None, names="value")   # redraw the same samples
refresh_wrong()
refresh_saved()
display(widgets.VBox([model_box,
                      widgets.HBox([n_box, cols_box, seed_box, steps_box]),
                      widgets.HBox([ll_box, button, redraw_button]),
                      view_box,
                      out,
                      widgets.HBox([pick_box, add_button, remove_button, clear_button]),
                      wrong_label,
                      dir_box,
                      widgets.HBox([name_box, save_button, save_wrong_button]),
                      widgets.HBox([saved_box, show_saved_button]),
                      msg]))
